# YOFC G.654.E — Executed validation (illustrative surrogate)

Reference: YOFC specifications supplied by user (MFD 11.4–12.2 µm, typical 11.8; typical Aeff 110 µm²; D1550 ≤22; dispersion slope ≤0.070). **2026-11-03**.

This notebook embeds the original `COMSOL/universal_silica_v01/universal_radial.py` radial P1 FEM source from LSCNS (blob `6feabf558786931e87d684375c6a4a6645aa79a9`), providing an independent run of the same underlying scalar LP01 method without network access. Model inputs core radius 5.4 µm and core Δn 0.0040 were **fitted** to typical MFD/Aeff, not obtained from YOFC; hence **fit ≠ independent physical validation**. 

MFD comparison uses Petermann-II (`MFD_PII_um`); the second-moment MFD is listed separately. n₂ is an assumed 2.2e-20 m²/W. Cable loss, cutoff, PMD, bends and real dopant dispersion are not modeled.

In [1]:
"""Research prototype v0.1: rotationally symmetric solid-silica fibers.

Scalar weak-guidance radial P1 finite element mode solver for LP01.
Materials are silica baseline Sellmeier plus *constant* delta_n offsets;
this is an illustrative index model, not a verified Ge/F doped Sellmeier law.
This solver does NOT model HCF/NANF, noncircular MCF, birefringence, loss, bends,
or fiber manufacturing variation. All transverse dimensions are in micrometres.
"""
from __future__ import annotations
from dataclasses import dataclass, replace, asdict
from typing import Sequence
import json
import numpy as np
from scipy.special import j0,j1,k0 as K0,k1 as K1
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh
from numpy.polynomial.legendre import leggauss

C0=299792458.0
_SELL_B=(0.6961663,0.4079426,0.8974794)
_SELL_C=(0.0684043**2,0.1162414**2,9.896161**2)

@dataclass(frozen=True)
class Model:
    profile: str = 'step'                  # step, graded, trench, multilayer
    wavelength_nm: float = 1550.0
    core_radius_um: float = 5.5
    delta_n_core: float = 0.0034785793846155
    outer_radius_um: float = 40.0       # computational, not physical cladding radius
    graded_alpha: float = 2.0
    trench_start_um: float = 7.0
    trench_end_um: float = 10.0
    trench_delta_n: float = -0.001
    # custom shells are (outer_radius_um, absolute delta_n relative to silica baseline).
    # First one starts at r=0; last listed shell may end before outer_radius_um.
    shells: tuple[tuple[float,float],...] = ((5.0,0.004), (7.0,0.0), (9.0,-0.0007))
    n2_m2_per_W: float = 2.2e-20
    def validate(self):
        if self.profile not in ('step','graded','trench','multilayer'):
            raise ValueError('profile must be step, graded, trench, or multilayer')
        for name in ('wavelength_nm','core_radius_um','outer_radius_um','graded_alpha','n2_m2_per_W'):
            x=float(getattr(self,name))
            if not np.isfinite(x) or x<=0: raise ValueError(f'{name} must be positive/finite')
        if not 250 < self.wavelength_nm < 3600: raise ValueError('wavelength outside silica Sellmeier domain')
        if self.outer_radius_um<=2.2*self.core_radius_um: raise ValueError('outer radius must exceed 2.2 x core radius')
        for name in ('delta_n_core','trench_delta_n'):
            if not np.isfinite(getattr(self,name)): raise ValueError(name)
        if self.profile=='trench' and not (self.core_radius_um < self.trench_start_um < self.trench_end_um < self.outer_radius_um):
            raise ValueError('trench radii must be core < start < end < outer')
        if self.profile=='multilayer':
            if not self.shells: raise ValueError('shells required')
            rr=np.array([x[0] for x in self.shells]); nn=np.array([x[1] for x in self.shells]);
            if np.any(~np.isfinite(rr)) or np.any(~np.isfinite(nn)) or np.any(np.diff(rr)<=0) or rr[0]<=0 or rr[-1]>=self.outer_radius_um:
                raise ValueError('shell boundaries must increase and lie below outer radius')
        return self

def silica_index(wavelength_um):
    w=np.asarray(wavelength_um,dtype=float)
    if np.any((w<=0.21)|(w>=3.7)): raise ValueError('silica Sellmeier only 0.21–3.7 µm')
    return np.sqrt(1+sum(b*w*w/(w*w-c) for b,c in zip(_SELL_B,_SELL_C)))

def boundaries(m:Model):
    m.validate()
    if m.profile=='trench': marks=[m.core_radius_um,m.trench_start_um,m.trench_end_um]
    elif m.profile=='multilayer': marks=[r for r,d in m.shells]
    else: marks=[m.core_radius_um]
    return np.r_[0.0,marks,m.outer_radius_um]

def delta_n_r(m:Model,r):
    r=np.asarray(r,dtype=float)
    if m.profile=='step': return np.where(r<m.core_radius_um,m.delta_n_core,0.0)
    if m.profile=='graded': return np.where(r<m.core_radius_um,m.delta_n_core*np.maximum(1-(r/m.core_radius_um)**m.graded_alpha,0),0.0)
    if m.profile=='trench':
        return np.where(r<m.core_radius_um,m.delta_n_core,
                        np.where((r>=m.trench_start_um)&(r<m.trench_end_um),m.trench_delta_n,0.0))
    out=np.zeros_like(r)
    left=0.0
    for right,dn in m.shells:
        out=np.where((r>=left)&(r<right),dn,out)
        left=right
    return out

def index_r(m:Model,r):
    return silica_index(m.wavelength_nm/1000)+delta_n_r(m,r)

def radial_grid(m:Model,elements=750):
    if not isinstance(elements,(int,np.integer)) or elements<80:raise ValueError('elements>=80')
    br=boundaries(m); intervals=np.diff(br)
    base=np.maximum(12,np.round(elements*intervals/np.sum(intervals)).astype(int))
    # allocate more near interfaces by maintaining at least 12 nodes per ring
    out=[np.linspace(br[i],br[i+1],int(base[i])+1)[:-1] for i in range(len(base))]
    return np.r_[np.concatenate(out),br[-1]]

def solve_lp01(m:Model,elements=750):
    m.validate(); r=radial_grid(m,elements)
    nodes=len(r); n_el=nodes-1
    t0,ww=leggauss(5); t=(t0+1)/2; wq=ww/2
    h=np.diff(r); rq=r[:-1,None]+h[:,None]*t
    weights=h[:,None]*wq
    basis=np.stack((1-t,t),axis=1)
    # quadratic-form angular integration has 2pi on both sides, cancels for eigenvalue.
    mass=np.einsum('eq,qi,qj->eij',weights*rq,basis,basis)
    deriv=(np.diff(r**2)/(2*h*h))[:,None,None]*np.array([[1.,-1.],[-1.,1.]])
    nvals=index_r(m,rq)
    n_mass=np.einsum('eq,eq,qi,qj->eij',weights*rq,nvals**2,basis,basis)
    k0=2*np.pi/(m.wavelength_nm/1000)
    mats=(k0*k0*n_mass-deriv,mass)
    edge=np.stack((np.arange(n_el),np.arange(1,nodes)),axis=1)
    ii=np.broadcast_to(edge[:,:,None],(n_el,2,2)).ravel()
    jj=np.broadcast_to(edge[:,None,:],(n_el,2,2)).ravel()
    A,B=[coo_matrix((v.ravel(),(ii,jj)),shape=(nodes,nodes)).tocsr()[:-1,:-1] for v in mats]
    n_max=float(np.max(index_r(m,rq)))
    eigen,ev=eigsh(A,k=1,M=B,sigma=(k0*n_max)**2+0.05,which='LM',tol=1e-10,v0=np.ones(nodes-1))
    beta2=float(eigen[0]); psi=np.r_[ev[:,0],0.0]
    if psi[0]<0:psi=-psi
    if beta2<=0: raise RuntimeError('nonpositive beta²')
    neff=float(np.sqrt(beta2)/k0)
    rel_res=float(np.linalg.norm(A@ev[:,0]-beta2*(B@ev[:,0]))/np.linalg.norm(beta2*(B@ev[:,0])))
    I=(psi[edge]@basis.T)**2
    wtr=weights*rq
    P=float(2*np.pi*np.sum(wtr*I)); Q=float(2*np.pi*np.sum(wtr*I**2));
    radial_moment=float(2*np.pi*np.sum(wtr*I*rq**2)/P)
    slope=np.diff(psi)/h
    nrm_grad=float(np.sum(weights*rq*slope[:,None]**2))
    nrm_psi=float(np.sum(weights*rq*I))
    mfd_pii=float(2*np.sqrt(2*nrm_psi/nrm_grad))
    aeff=P**2/Q
    tail_power=float(2*np.pi*np.sum(wtr[rq[:,0]>0.8*m.outer_radius_um]*I[rq[:,0]>0.8*m.outer_radius_um])/P)
    return {'model':m,'neff':neff,'Aeff_um2':aeff,'MFD_2nd_um':2*np.sqrt(2*radial_moment),'MFD_PII_um':mfd_pii,
            'gamma_Wkm':2*np.pi/(m.wavelength_nm*1e-9)*m.n2_m2_per_W/(aeff*1e-12)*1000,
            'residual':rel_res,'tail_power':tail_power,'guided_proxy':bool(neff>float(silica_index(m.wavelength_nm/1000))),
            'r_um':r,'field':psi/np.sqrt(P),'elements':n_el}

def bessel_step_oracle(m:Model):
    if m.profile!='step':raise ValueError('Bessel oracle only valid for step-index')
    k0=2*np.pi/(m.wavelength_nm/1000)
    ncl=float(silica_index(m.wavelength_nm/1000)); nco=ncl+m.delta_n_core
    if nco<=ncl:raise ValueError('index-guided core required')
    V=k0*m.core_radius_um*np.sqrt(nco*nco-ncl*ncl)
    if not (0<V<2.40482555769):raise ValueError('LP01 reference restricted to V<2.405')
    def fn(u):
        w=np.sqrt(V*V-u*u)
        return u*j1(u)/j0(u)-w*K1(w)/K0(w)
    u=brentq(fn,max(1e-8,V*1e-6),V*(1-1e-10),xtol=3e-13)
    neff=float(np.sqrt(nco*nco-(u/(k0*m.core_radius_um))**2))
    return {'neff':neff,'V':V}

def spectral_neff(m:Model,lambda_nm,elements=750):
    return solve_lp01(replace(m,wavelength_nm=float(lambda_nm)),elements)['neff']

def chromatic_dispersion(m:Model,elements=750,h_nm=10.0):
    # D ps/(nm.km) = -(lambda/c) d2 n_eff/d(lambda_um)2 * 1e12
    # All eigenvalue samples computed on identical radial mesh (wavelength-independent geometry).
    w=m.wavelength_nm/1000; h=h_nm/1000
    n=np.array([spectral_neff(m,(w+j*h)*1000,elements) for j in (-2,-1,0,1,2)])
    d2=(-n[4]+16*n[3]-30*n[2]+16*n[1]-n[0])/(12*h*h)
    return float(-w/C0*1e12*d2)

def serialize(sol):
    return {k:(asdict(v) if isinstance(v,Model) else float(v) if isinstance(v,(np.floating,np.integer)) else v)
            for k,v in sol.items() if k not in ('r_um','field')}

PRESETS={
 'G.654.E-like (illustrative)':dict(profile='step',core_radius_um=5.5,delta_n_core=0.0034785793846155),
 'G.652.D-like (illustrative)':dict(profile='step',core_radius_um=4.1,delta_n_core=0.005),
 'Graded core (illustrative)':dict(profile='graded',core_radius_um=5.0,delta_n_core=0.006,graded_alpha=2.0),
 'Trench-assisted (illustrative)':dict(profile='trench',core_radius_um=4.1,delta_n_core=0.005,trench_start_um=6.0,trench_end_um=9.0,trench_delta_n=-0.001),
 'Multilayer (illustrative)':dict(profile='multilayer',outer_radius_um=40.0,shells=((4.0,0.006),(5.0,0.002),(7.0,0.0),(9.0,-0.0008)))
}


In [2]:
from dataclasses import replace, asdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
m=Model(profile='step',wavelength_nm=1550.,core_radius_um=5.4,
        delta_n_core=0.004,outer_radius_um=40.,n2_m2_per_W=2.2e-20)
n_elements=750
sol=solve_lp01(m,elements=n_elements)
D1550=chromatic_dispersion(m,elements=n_elements,h_nm=10.)
print('MFD_PII_um:',round(sol['MFD_PII_um'],6))
print('MFD_2nd_um:',round(sol['MFD_2nd_um'],6))
print('Aeff_um2:',round(sol['Aeff_um2'],6))
print('D1550_ps_nm_km:',round(D1550,6))
print('neff:',round(sol['neff'],9))
print('gamma_Wkm:',round(sol['gamma_Wkm'],6))

MFD_PII_um: 11.834031
MFD_2nd_um: 12.085655
Aeff_um2: 110.367841
D1550_ps_nm_km: 20.103068
neff: 1.446099217
gamma_Wkm: 0.808032


In [3]:
comparison=pd.DataFrame([
 {'parameter':'MFD_PII @1550','YOFC_reference':'11.4–12.2 (typical 11.8) µm','simulation':sol['MFD_PII_um'],'typical_relative_error_pct':(sol['MFD_PII_um']/11.8-1)*100},
 {'parameter':'Aeff @1550','YOFC_reference':'typical 110 µm² (not acceptance tolerance)','simulation':sol['Aeff_um2'],'typical_relative_error_pct':(sol['Aeff_um2']/110-1)*100},
 {'parameter':'D @1550','YOFC_reference':'≤22 ps/(nm·km)','simulation':D1550,'typical_relative_error_pct':float('nan')}
])
print(comparison.to_string(index=False,float_format=lambda x:f'{x:.5f}'))
print('Caution: target-fitted agreement; not YOFC-specific physical validation')

             parameter                              YOFC_reference  simulation  typical_relative_error_pct
        MFD_PII @1550                11.4–12.2 (typical 11.8) µm    11.83403                     0.28840
           Aeff @1550 typical 110 µm² (not acceptance tolerance)   110.36784                     0.33440
              D @1550                              ≤22 ps/(nm·km)    20.10307                         nan
Caution: target-fitted agreement; not YOFC-specific physical validation


In [4]:
mesh_rows=[]
for n in (300,500,750,1000):
    q=solve_lp01(m,elements=n)
    mesh_rows.append(dict(elements=n,MFD_PII_um=q['MFD_PII_um'],MFD_2nd_um=q['MFD_2nd_um'],
                     Aeff_um2=q['Aeff_um2'],D1550_ps_nm_km=chromatic_dispersion(m,elements=n,h_nm=10),
                     eig_residual=q['residual']))
mesh_df=pd.DataFrame(mesh_rows)
print(mesh_df[['elements','MFD_PII_um','Aeff_um2','D1550_ps_nm_km']].to_string(index=False,float_format=lambda x:f'{x:.6f}'))
print('Aeff 750->1000 relative difference (%):',abs(mesh_df.iloc[2].Aeff_um2/mesh_df.iloc[3].Aeff_um2-1)*100)

 elements  MFD_PII_um   Aeff_um2  D1550_ps_nm_km
      300   11.834270  110.373769        20.102564
      500   11.834089  110.369281        20.102950
      750   11.834031  110.367841        20.103068
     1000   11.834011  110.367343        20.103109
Aeff 750->1000 relative difference (%): 0.0004509


In [5]:
spectrum_rows=[]
for wave in range(1530,1621,10):
    mm=replace(m,wavelength_nm=float(wave))
    a=solve_lp01(mm,elements=n_elements)
    spectrum_rows.append(dict(wavelength_nm=wave,MFD_PII_um=a['MFD_PII_um'],Aeff_um2=a['Aeff_um2'],
                              D_ps_nm_km=chromatic_dispersion(mm,elements=n_elements,h_nm=5),neff=a['neff']))
spec_df=pd.DataFrame(spectrum_rows)
slope=(spec_df.loc[spec_df.wavelength_nm==1560,'D_ps_nm_km'].iloc[0]-spec_df.loc[spec_df.wavelength_nm==1540,'D_ps_nm_km'].iloc[0])/20
print(spec_df[['wavelength_nm','MFD_PII_um','Aeff_um2','D_ps_nm_km']].to_string(index=False,float_format=lambda x:f'{x:.6f}'))
print('D slope near 1550 (ps/nm²/km):',round(slope,7),'(YOFC limit ≤0.070)')

 wavelength_nm  MFD_PII_um   Aeff_um2  D_ps_nm_km
          1530   11.729991  108.713353   18.907629
          1540   11.781837  109.535729   19.508841
          1550   11.834031  110.367841   20.103067
          1560   11.886578  111.209840   20.690610
          1570   11.939480  112.061883   21.271761
          1580   11.992743  112.924126   21.846802
          1590   12.046368  113.796731   22.416007
          1600   12.100361  114.679861   22.979637
          1610   12.154726  115.573681   23.537947
          1620   12.209466  116.478363   24.091183
D slope near 1550 (ps/nm²/km): 0.0590884 (YOFC limit ≤0.070)


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(spec_df.wavelength_nm,spec_df.MFD_PII_um,'o-')
ax[0].scatter([1550],[11.8],label='YOFC MFD typical @1550 nm')
ax[0].set(xlabel='Wavelength (nm)',ylabel='MFD PII (µm)',title='Mode field diameter')
ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(spec_df.wavelength_nm,spec_df.D_ps_nm_km,'o-')
ax[1].scatter([1550],[22],label='YOFC D upper bound @1550 nm')
ax[1].set(xlabel='Wavelength (nm)',ylabel='D (ps/nm/km)',title='Chromatic dispersion')
ax[1].legend(); ax[1].grid(alpha=.3)
fig.suptitle('YOFC G.654.E illustrative step-index fit (not measured vendor profile)')
plt.tight_layout(); plt.show()

In [6]:
outdir=Path('validation_outputs');outdir.mkdir(exist_ok=True)
comparison.to_csv(outdir/'comparison.csv',index=False)
mesh_df.to_csv(outdir/'mesh_convergence.csv',index=False)
spec_df.to_csv(outdir/'spectrum.csv',index=False)
summary={'method':'scalar LP01 radial P1 FEM, fitted illustrative step-index',
         'core_radius_um':m.core_radius_um,'core_delta_n':m.delta_n_core,
         'elements':n_elements,'MFD_PII_um':float(sol['MFD_PII_um']),
         'MFD_2nd_um':float(sol['MFD_2nd_um']),'Aeff_um2':float(sol['Aeff_um2']),
         'D1550_ps_nm_km':float(D1550),'slope_ps_nm2_km':float(slope)}
(outdir/'summary.json').write_text(json.dumps(summary,indent=2))
print('Generated comparison.csv, mesh_convergence.csv, spectrum.csv, summary.json')

Generated comparison.csv, mesh_convergence.csv, spectrum.csv, summary.json


## Limitations

The 1550 nm mode-field and effective-area outputs are **fitted** rather than independent blind predictions. The upper-bound checks for D and slope apply only to the surrogate model. The notebook does **not** assess attenuation, cable cutoff, PMD, macro-bending, real dopant dispersion or product compliance. COMSOL Wave Optics was not run. An actual measured refractive-index profile and measurement conventions would be needed for product-level validation.